# Demonstrasi Keamanan JSON Web Token (JWT)
Notebook ini menunjukkan secara langsung simulasi proses otentikasi dan kegagalan token akibat manipulasi (Tampering) dengan menggunakan modul backend asli.

In [6]:
import sys
import os
import json
import base64
from fastapi.testclient import TestClient

# Add backend directory to path
backend_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'backend'))
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)

from app.main import app
from app.auth import AuthService

client = TestClient(app)

print("="*70)
print("PENGUJIAN DIGITAL SIGNATURE & JWT TAMPERING MENGGUNAKAN SERVER ASLI")
print("="*70)

# Tahap 1: Mendapatkan Token Valid
print("\n[TAHAP 1] Server menerbitkan Token Otentikasi Pengguna")
valid_token = AuthService.create_access_token(
    data={"sub": "mhs_arip", "role": "mahasiswa"}
)
print(f"   >>> Token ter-generate: {valid_token[:30]}...{valid_token[-30:]}")

# Tahap 2: Akses Endpoint dengan Token Valid
print("\n[TAHAP 2] Klien dengan Otoritas Sah Mengakses REST API (/auth/me)")
response = client.get("/auth/me", headers={"Authorization": f"Bearer {valid_token}"})
print(f"   HTTP Status: {response.status_code}")
if response.status_code == 200:
    user_data = response.json()
    print(f"   Response Body: (Berhasil mengakses profil, Nama: {user_data.get('full_name')})")
else:
    print(f"   Response Body: {response.json()}")

# Tahap 3: Tampering Token
print("\n[TAHAP 3] Entitas Tidak Sah Melakukan Manipulasi Token (Privilege Escalation)")
parts = valid_token.split(".")
header, payload, signature = parts
payload_padded = payload + '=' * (-len(payload) % 4)
decoded_payload = json.loads(base64.urlsafe_b64decode(payload_padded).decode('utf-8'))

print(f"   [UNAUTHORIZED_MODIFICATION] Mengubah role dari '{decoded_payload.get('role')}' menjadi 'admin'...")
decoded_payload["role"] = "admin"
tampered_payload = base64.urlsafe_b64encode(json.dumps(decoded_payload).encode('utf-8')).decode('utf-8').rstrip('=')
tampered_token = f"{header}.{tampered_payload}.{signature}"

# Tahap 4: Akses Endpoint dengan Token Palsu
print("\n[TAHAP 4] Klien Tidak Sah Mengakses REST API (/auth/me) menggunakan Token Palsu")
response_tampered = client.get("/auth/me", headers={"Authorization": f"Bearer {tampered_token}"})
print(f"   HTTP Status: {response_tampered.status_code}")
print(f"   Response Body: {response_tampered.json()}")

print("\n" + "="*70)
print("KESIMPULAN:")
if response_tampered.status_code == 401:
    print("Server secara otomatis menolak token yang dimanipulasi dengan status 401 Unauthorized.")
    print("Mekanisme Digital Signature pada server terbukti aman dan mencegah Tampering.")
print("="*70)


PENGUJIAN DIGITAL SIGNATURE & JWT TAMPERING MENGGUNAKAN SERVER ASLI

[TAHAP 1] Server menerbitkan Token Otentikasi Pengguna
   >>> Token ter-generate: eyJhbGciOiJIUzI1NiIsInR5cCI6Ik...2dOe-4JKR7cvczLmTHVU7rMq4kLl4w

[TAHAP 2] Klien dengan Otoritas Sah Mengakses REST API (/auth/me)
   HTTP Status: 200
   Response Body: (Berhasil mengakses profil, Nama: Arief Abdul Rahman)

[TAHAP 3] Entitas Tidak Sah Melakukan Manipulasi Token (Privilege Escalation)
   [UNAUTHORIZED_MODIFICATION] Mengubah role dari 'mahasiswa' menjadi 'admin'...

[TAHAP 4] Klien Tidak Sah Mengakses REST API (/auth/me) menggunakan Token Palsu
   HTTP Status: 401
   Response Body: {'detail': 'Token tidak valid atau sudah kedaluwarsa.'}

KESIMPULAN:
Server secara otomatis menolak token yang dimanipulasi dengan status 401 Unauthorized.
Mekanisme Digital Signature pada server terbukti aman dan mencegah Tampering.
